# FINAL 03 — RQ1 Linguistic Subgroup Performance

**Thesis:** An Ethical Risk Assessment of Sentiment Analysis Models in Social Media Monitoring with a Focus on Bias, Inclusivity, and Misclassification Patterns  
**Student:** Abhishek Tomar | PN1196973 | LJMU | October 2026

**Purpose:** Answer RQ1 from the locked final TweetEval predictions without any retraining or subgroup-threshold tuning.

> **Final-run rule:** TweetEval test data is held out from model/feature/hyperparameter selection. External datasets are never used for source-model selection. Linguistic subgroup thresholds are frozen and are not tuned after observing subgroup sizes.

In [5]:
!pip install -q emoji vaderSentiment pandas pyarrow scikit-learn scipy google-api-python-client

from google.colab import drive, auth
drive.mount('/content/drive')

import io
import sys
from pathlib import Path
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Shared final-run utility stored in the Drive Experiments folder.
UTIL_FILE_ID = "14KfwtUf2zSd_jlW7i-6K8XgBJ62Aqk02"
UTIL_LOCAL = Path("/content/final_thesis_utils.py")

def load_final_thesis_utils():
    if not UTIL_LOCAL.exists() or UTIL_LOCAL.stat().st_size == 0:
        auth.authenticate_user()
        creds, _ = google.auth.default()
        service = build("drive", "v3", credentials=creds, cache_discovery=False)
        request = service.files().get_media(fileId=UTIL_FILE_ID)
        with io.FileIO(str(UTIL_LOCAL), "wb") as fh:
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done:
                status, done = downloader.next_chunk()
                if status:
                    print(f"Downloading final_thesis_utils.py: {int(status.progress()*100)}%")

    assert UTIL_LOCAL.exists() and UTIL_LOCAL.stat().st_size > 0, \
        "final_thesis_utils.py could not be downloaded."

    if "/content" not in sys.path:
        sys.path.insert(0, "/content")

    import importlib
    import final_thesis_utils as ftu
    ftu = importlib.reload(ftu)

    RUN_ROOT = Path("/content/drive/MyDrive/LJMU_FINAL_EXPERIMENT_RUN")
    ftu.FINAL_ROOT = RUN_ROOT
    ftu.FINAL = {
        "root": RUN_ROOT,
        "data": RUN_ROOT / "data",
        "models": RUN_ROOT / "models",
        "predictions": RUN_ROOT / "predictions",
        "results": RUN_ROOT / "results",
        "figures": RUN_ROOT / "figures",
        "manifests": RUN_ROOT / "manifests",
    }
    return ftu

ftu = load_final_thesis_utils()

globals().update({k: v for k, v in vars(ftu).items() if not k.startswith("_")})
FINAL_ROOT = ftu.FINAL_ROOT
FINAL = ftu.FINAL
setup_final_dirs()

print("✅ final_thesis_utils imported successfully")
print("✅ Final run root:", FINAL_ROOT)

import pandas as pd, numpy as np

from sklearn.metrics import precision_recall_fscore_support

PRED_PATH = FINAL["predictions"] / "FINAL_SELECTED_TweetEval.parquet"
assert PRED_PATH.exists(), f"Missing FINAL_02 prediction file: {PRED_PATH}"

pred = pd.read_parquet(PRED_PATH)

required_cols = {
    "row_id","text","y_true","y_pred","true_label","pred_label",
    "subgroup","model","dataset","confidence","correct"
}
missing_cols = required_cols - set(pred.columns)
assert not missing_cols, f"Prediction file is missing required columns: {sorted(missing_cols)}"
assert len(pred) == 12284, f"Expected 12,284 TweetEval test rows, found {len(pred):,}"

expected_subgroups = {
    "reference-unmarked": 10388,
    "other": 1112,
    "emoji-heavy": 710,
    "slang-heavy": 60,
    "sarcasm-indicated": 14,
}
actual_subgroups = pred["subgroup"].value_counts().to_dict()
assert actual_subgroups == expected_subgroups, (
    f"Subgroup counts do not match FINAL_01. Expected {expected_subgroups}; "
    f"found {actual_subgroups}"
)

print(f"✅ Loaded FINAL_02 held-out predictions: {len(pred):,} rows")
print(pred["subgroup"].value_counts())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ final_thesis_utils imported successfully
✅ Final run root: /content/drive/MyDrive/LJMU_FINAL_EXPERIMENT_RUN
✅ Loaded FINAL_02 held-out predictions: 12,284 rows
subgroup
reference-unmarked    10388
other                  1112
emoji-heavy             710
slang-heavy              60
sarcasm-indicated        14
Name: count, dtype: int64


In [6]:
perf=subgroup_performance(pred)
ref_f1=float(perf.loc[perf["Subgroup"]==REFERENCE_GROUP,"Macro F1"].iloc[0])
perf["Macro F1 Gap vs Reference"]=perf["Macro F1"]-ref_f1
perf.to_csv(FINAL["results"]/"F05_RQ1_subgroup_performance.csv",index=False)
display(perf.sort_values("Subgroup"))

,Subgroup,N,Accuracy,Macro F1,Macro F1 CI Low,Macro F1 CI High,Misclassification Rate,Interpretation Status,Macro F1 Gap vs Reference
0,emoji-heavy,710,0.652113,0.635388,0.598448,0.672985,0.347887,stable_quantitative,0.038462
1,other,1112,0.661871,0.591532,0.556396,0.626407,0.338129,stable_quantitative,-0.005394
2,reference-unmarked,10388,0.614748,0.596926,0.587504,0.606141,0.385252,stable_quantitative,0.000000
3,sarcasm-indicated,14,0.785714,0.789744,0.475308,1.000000,0.214286,exploratory_descriptive_only,0.192817
4,slang-heavy,60,0.616667,0.619431,0.498602,0.731746,0.383333,stable_quantitative,0.022505


In [7]:
# Per-class performance by subgroup
rows=[]
for sg,part in pred.groupby("subgroup",observed=True):
    p,r,f,s=precision_recall_fscore_support(part.y_true,part.y_pred,labels=[0,1,2],zero_division=0)
    for i,name in ID_TO_LABEL.items():
        rows.append({"Subgroup":sg,"Class":name,"Precision":p[i],"Recall":r[i],"F1":f[i],"Support":int(s[i])})
class_perf=pd.DataFrame(rows)
class_perf.to_csv(FINAL["results"]/"F05_RQ1_subgroup_class_performance.csv",index=False)
display(class_perf)

,Subgroup,Class,Precision,Recall,F1,Support
0,emoji-heavy,negative,0.737374,0.486667,0.586345,150
1,emoji-heavy,neutral,0.532203,0.646091,0.583643,243
2,emoji-heavy,positive,0.737342,0.735016,0.736177,317
3,other,negative,0.587719,0.347150,0.436482,193
4,other,neutral,0.727941,0.759202,0.743243,652
5,other,positive,0.547170,0.651685,0.594872,267
6,reference-unmarked,negative,0.714769,0.468256,0.565829,3607
7,reference-unmarked,neutral,0.621688,0.707327,0.661748,5009
8,reference-unmarked,positive,0.496131,0.651242,0.563202,1772
9,sarcasm-indicated,negative,0.800000,0.800000,0.800000,5


### Interpretation guard
`reference-unmarked` is a comparison group defined by the absence of the three focal markers plus a minimum word count. It is not a demographic privileged group. `sarcasm-indicated` results below the frozen minimum sample size are descriptive/exploratory.